# Module 6 Instructor Colab Notebook: Integration and Continuous Distributions

## A Mathematical Bridge to Artificial Intelligence

Integration combines infinitely many small contributions. It can recover accumulated change, measure signed area, normalize a continuous probability density, and compute an expected value. These are not separate tricks: each uses the same limiting idea of adding increasingly fine pieces.

For AI, integration explains why the total mass of a probability density is one and why population loss is an expected loss over possible data. The notebook emphasizes a crucial distinction: a density value is not itself a probability; probability is obtained by integrating density over a range.

**Recommended pacing:** three class meetings. The Riemann-sum derivation and integration-by-parts calculation may be used as instructor-led extensions.


> **How to use this notebook in class**
>
> Ask students to estimate area and probability visually before calculating. Keep units visible: density has probability per input unit, while an integral of density is dimensionless probability. When using expected loss, distinguish the ideal population distribution from the finite training sample.

## Notebook map

| Section | Main mathematical ideas | Direct AI connection |
|---|---|---|
| 1 | Antiderivatives and indefinite integrals | Recovering an accumulated quantity from its rate |
| 2 | Definite integrals, Riemann sums, and the FTC | Numerical aggregation and area |
| 3 | Improper integrals | Distributions supported on infinite intervals |
| 4 | PDFs and CDFs | Continuous probabilistic model outputs |
| 5 | Expected value and expected loss | Population training objectives |
| 6 | Numerical and sample approximations | Estimating integrals from grids or data |
| Integrated lab | Optimal constant prediction | Minimize expected squared loss |


In [ ]:
# Core setup: compatible with Google Colab and standard Jupyter.
import math
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

np.set_printoptions(precision=5, suppress=True)
plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

try:
    import ipywidgets as widgets
    from ipywidgets import interact, FloatSlider, IntSlider
    WIDGETS_AVAILABLE = True
except Exception:
    WIDGETS_AVAILABLE = False

print("Setup complete.")
print("Interactive widgets available:", WIDGETS_AVAILABLE)


In [ ]:
# Reusable instructor-solution helper.
def reveal_markdown(button_text, solution_markdown):
    if not WIDGETS_AVAILABLE:
        display(Markdown("**Instructor solution**"))
        display(Markdown(solution_markdown))
        return

    button = widgets.Button(description=button_text, button_style="info")
    output = widgets.Output()
    state = {"open": False}

    def toggle(_):
        with output:
            output.clear_output()
            state["open"] = not state["open"]
            if state["open"]:
                display(Markdown(solution_markdown))
                button.description = "Hide instructor solution"
            else:
                button.description = button_text

    button.on_click(toggle)
    display(button, output)


def tangent_line(function, derivative, point, x_values):
    return function(point) + derivative(point) * (x_values - point)


## Module learning objectives

By the end of the module, students should be able to:

1. interpret an antiderivative and include the constant of integration;
2. connect Riemann sums, signed area, and definite integrals;
3. use both parts of the Fundamental Theorem of Calculus;
4. determine convergence of basic improper integrals;
5. verify whether a function is a probability density;
6. move between a PDF and a CDF;
7. compute expectations of functions of a continuous random variable; and
8. distinguish population expected loss from empirical average loss.

## Prerequisite checkpoint

Students should know functions, limits, derivatives, exponentials, and basic algebra. Module 5 supplies the required derivative background.


---

# Section 1: Antiderivatives and Indefinite Integrals

## 1. Accumulation reverses differentiation

Suppose a rate is known and we want the quantity that produced it. An **antiderivative** of a function $$f$$ is any function $$F$$ satisfying

$$
F'(x)=f(x).
$$

The **indefinite integral** denotes the family of all antiderivatives:

$$
\int f(x)\,dx=F(x)+C.
$$

The symbol $$\int$$ is the integral sign, $$f(x)$$ is the **integrand**, $$x$$ is the **variable of integration**, and $$C$$ is the **constant of integration**.

The differential symbol $$dx$$ identifies the variable being accumulated. At this stage it should be read as part of the integral notation.


## 2. Why the constant is necessary

All constant shifts have the same derivative:

$$
\frac{d}{dx}\bigl(F(x)+C\bigr)=F'(x)+0=f(x).
$$

For example,

$$
\frac{d}{dx}(x^2)=2x,
$$

$$
\frac{d}{dx}(x^2+5)=2x,
$$

and

$$
\frac{d}{dx}(x^2-100)=2x.
$$

Therefore

$$
\boxed{\int 2x\,dx=x^2+C.}
$$

An initial or boundary condition can select one member of this family.


## 3. Basic antiderivative rules

For $$n\ne -1,$$

$$
\int x^n\,dx=\frac{x^{n+1}}{n+1}+C.
$$

This is the reverse of the power rule because

$$
\frac{d}{dx}\left(\frac{x^{n+1}}{n+1}\right)
=\frac{n+1}{n+1}x^n=x^n.
$$

Other basic rules include

$$
\int e^x\,dx=e^x+C,
$$

$$
\int \frac{1}{x}\,dx=\ln|x|+C,
\qquad x\ne 0,
$$

and linearity:

$$
\int \bigl(af(x)+bg(x)\bigr)\,dx
=a\int f(x)\,dx+b\int g(x)\,dx.
$$


## 4. Detailed antiderivative example

Compute

$$
\int \left(6x^2-4x+3\right)\,dx.
$$

Integrate term by term:

$$
\int 6x^2\,dx
=6\left(\frac{x^3}{3}\right)
=2x^3,
$$

$$
\int (-4x)\,dx
=-4\left(\frac{x^2}{2}\right)
=-2x^2,
$$

and

$$
\int 3\,dx=3x.
$$

Thus

$$
\boxed{\int \left(6x^2-4x+3\right)\,dx
=2x^3-2x^2+3x+C.}
$$

Differentiate the answer to verify:

$$
\frac{d}{dx}\left(2x^3-2x^2+3x+C\right)
=6x^2-4x+3.
$$


## 5. Recovering a quantity from a rate

Suppose a training diagnostic changes at rate

$$
r(t)=6t-4
$$

units per hour. If its value at time zero is $$10,$$ then

$$
Q(t)=\int (6t-4)\,dt=3t^2-4t+C.
$$

Apply the initial condition:

$$
Q(0)=C=10.
$$

Therefore

$$
\boxed{Q(t)=3t^2-4t+10.}
$$

The antiderivative supplies the family, and the observed value selects the appropriate constant.


> **Common mistakes in Section 1**
>
> - Forgetting the constant of integration.
> - Decreasing the power as if differentiating instead of increasing it.
> - Applying the power formula with exponent negative one.
> - Omitting the absolute value in the logarithmic antiderivative.
> - Treating an indefinite integral as a single number.

## Section 1 terminology

| Term | Meaning |
|---|---|
| Antiderivative | Function whose derivative is the given integrand |
| Indefinite integral | Family of all antiderivatives |
| Integrand | Function being integrated |
| Variable of integration | Variable indicated by the differential symbol |
| Constant of integration | Arbitrary vertical shift shared by antiderivatives |
| Initial condition | Known value used to determine the constant |


## Section 1 guided worksheet

1. Compute

   $$
   \int (8x^3-6x+5)\,dx.
   $$

2. Verify the result by differentiation.

3. Find $$F$$ if

   $$
   F'(x)=4x-3
   $$

   and

   $$
   F(0)=7.
   $$

4. Explain why an indefinite integral needs $$C$$ but a definite integral does not.


In [ ]:
reveal_markdown("Open Section 1 solutions", r"""
1. Integrate each term:

   $$
   \int 8x^3\,dx=8\left(\frac{x^4}{4}\right)=2x^4,
   $$

   $$
   \int (-6x)\,dx=-6\left(\frac{x^2}{2}\right)=-3x^2,
   $$

   and

   $$
   \int 5\,dx=5x.
   $$

   Hence

   $$
   \boxed{2x^4-3x^2+5x+C}.
   $$

2. Differentiate:

   $$
   \frac{d}{dx}(2x^4-3x^2+5x+C)
   =8x^3-6x+5.
   $$

3. First,

   $$
   F(x)=2x^2-3x+C.
   $$

   Since

   $$
   F(0)=C=7,
   $$

   the required function is

   $$
   \boxed{F(x)=2x^2-3x+7.}
   $$

4. An indefinite integral represents a family of antiderivatives that differ by constants. A definite integral evaluates a net accumulation between fixed bounds; the constant cancels when endpoint values are subtracted.
""")


---

# Section 2: Definite Integrals, Riemann Sums, and the Fundamental Theorem

## 1. Adding many small pieces

Partition an interval from $$a$$ to $$b$$ into $$n$$ subintervals of width

$$
\Delta x=\frac{b-a}{n}.
$$

Choose a sample point $$x_i^*$$ in each subinterval. The **Riemann sum**

$$
\sum_{i=1}^{n}f(x_i^*)\Delta x
$$

adds rectangle contributions. If these sums approach a finite limit as the largest subinterval width approaches zero, the **definite integral** is

$$
\int_a^b f(x)\,dx
=\lim_{n\to\infty}\sum_{i=1}^{n}f(x_i^*)\Delta x.
$$

Unlike an indefinite integral, a definite integral is a number.


## 2. Derive the integral of a square from a right-endpoint sum

Compute

$$
\int_0^2 x^2\,dx.
$$

Divide the interval into $$n$$ equal pieces:

$$
\Delta x=\frac{2}{n}.
$$

The right endpoint of the $$i$$th interval is

$$
x_i=\frac{2i}{n}.
$$

The right-endpoint sum is

$$
S_n=\sum_{i=1}^{n}x_i^2\Delta x.
$$

Substitute:

$$
S_n=\sum_{i=1}^{n}\left(\frac{2i}{n}\right)^2\frac{2}{n}
=\frac{8}{n^3}\sum_{i=1}^{n}i^2.
$$

Use

$$
\sum_{i=1}^{n}i^2=\frac{n(n+1)(2n+1)}{6}.
$$

Then

$$
S_n=\frac{8}{n^3}\frac{n(n+1)(2n+1)}{6}.
$$

Cancel one factor of $$n$$:

$$
S_n=\frac{4(n+1)(2n+1)}{3n^2}.
$$

Divide numerator and denominator by $$n^2$$:

$$
S_n=\frac{4}{3}\left(1+\frac{1}{n}\right)
\left(2+\frac{1}{n}\right).
$$

Taking the limit gives

$$
\boxed{\int_0^2x^2\,dx
=\lim_{n\to\infty}S_n
=\frac{4}{3}(1)(2)
=\frac{8}{3}.}
$$


In [ ]:
def right_sum_square(n):
    delta = 2/n
    right_endpoints = delta*np.arange(1, n+1)
    return np.sum(right_endpoints**2)*delta

n_values_riemann = np.array([2, 4, 8, 20, 100, 1000])
sums_riemann = np.array([right_sum_square(int(n)) for n in n_values_riemann])
for n, value in zip(n_values_riemann, sums_riemann):
    print(f"n = {n:4d}, right sum = {value:.8f}, error = {value-8/3:.8f}")


In [ ]:
n_rectangles = 8
edges_rectangles = np.linspace(0, 2, n_rectangles+1)
right_rectangles = edges_rectangles[1:]
width_rectangles = 2/n_rectangles
x_curve_area = np.linspace(0, 2, 400)

fig, ax = plt.subplots()
ax.plot(x_curve_area, x_curve_area**2, linewidth=2, label="f(x) = x squared")
ax.bar(edges_rectangles[:-1], right_rectangles**2, width=width_rectangles,
       align="edge", alpha=0.35, edgecolor="navy", label="right rectangles")
ax.set_xlabel("x")
ax.set_ylabel("height")
ax.set_title("A Riemann sum adds thin rectangle areas")
ax.legend()
plt.show()


## 3. Signed area

When $$f(x)\geq 0,$$ the definite integral is ordinary area under the graph. When $$f(x)<0,$$ contributions are negative. Thus an integral measures **signed area**:

$$
\int_a^b f(x)\,dx
=\text{area above the axis}-\text{area below the axis}.
$$

For

$$
f(x)=x
$$

on the symmetric interval from $$-1$$ to $$1,$$ the triangular areas cancel:

$$
\int_{-1}^{1}x\,dx=0.
$$

The total geometric area is instead

$$
\int_{-1}^{1}|x|\,dx=1.
$$


## 4. Fundamental Theorem of Calculus

The theorem connects accumulation and differentiation.

### Part I: accumulation functions

If

$$
A(x)=\int_a^x f(t)\,dt,
$$

then, under continuity,

$$
A'(x)=f(x).
$$

The derivative of accumulated area is the current accumulation rate.

### Part II: evaluating definite integrals

If $$F'(x)=f(x),$$ then

$$
\boxed{\int_a^b f(x)\,dx=F(b)-F(a).}
$$

The right side is often written

$$
\left[F(x)\right]_a^b.
$$


## 5. Detailed FTC example

Compute

$$
\int_1^3(2x+1)\,dx.
$$

An antiderivative is

$$
F(x)=x^2+x.
$$

Evaluate at the upper endpoint:

$$
F(3)=3^2+3=12.
$$

Evaluate at the lower endpoint:

$$
F(1)=1^2+1=2.
$$

Subtract lower from upper:

$$
\boxed{\int_1^3(2x+1)\,dx=12-2=10.}
$$

The constant of integration is unnecessary because

$$
(F(b)+C)-(F(a)+C)=F(b)-F(a).
$$


## 6. Numerical integration

When no convenient antiderivative is available, a computer can approximate the integral. The **trapezoidal rule** replaces each curved piece by a trapezoid. For equally spaced grid points,

$$
\int_a^b f(x)\,dx
\approx \sum_{i=0}^{n-1}\frac{f(x_i)+f(x_{i+1})}{2}\Delta x.
$$

This is a deterministic approximation error, separate from sampling uncertainty in data.


In [ ]:
def trapezoid_integral(x_values, y_values):
    widths = np.diff(x_values)
    average_heights = (y_values[:-1] + y_values[1:])/2
    return np.sum(widths*average_heights)

grid_sizes_trap = np.array([3, 5, 11, 21, 101, 1001])
approximations_trap = []
for grid_size in grid_sizes_trap:
    x_grid = np.linspace(0, 2, int(grid_size))
    approximations_trap.append(trapezoid_integral(x_grid, x_grid**2))
approximations_trap = np.array(approximations_trap)

fig, ax = plt.subplots()
ax.loglog(grid_sizes_trap-1, np.abs(approximations_trap-8/3), marker="o")
ax.set_xlabel("number of subintervals")
ax.set_ylabel("absolute error")
ax.set_title("Trapezoidal approximation improves on a finer grid")
plt.show()


> **Common mistakes in Section 2**
>
> - Confusing a Riemann sum with its limiting integral.
> - Forgetting the rectangle width.
> - Treating signed area as total geometric area.
> - Subtracting endpoints in the wrong order.
> - Adding a constant to a definite-integral answer.
> - Assuming a numerical approximation is exact.

## Section 2 terminology

| Term | Meaning |
|---|---|
| Partition | Division of an interval into subintervals |
| Riemann sum | Sum of height-times-width contributions |
| Definite integral | Limit of refining Riemann sums |
| Signed area | Positive area above and negative area below the axis |
| Fundamental Theorem of Calculus | Connection between derivatives and definite integrals |
| Trapezoidal rule | Numerical approximation using trapezoid areas |


## Section 2 guided worksheet

1. Write the right-endpoint Riemann sum for

   $$
   \int_0^1x\,dx
   $$

   using $$n$$ equal subintervals.

2. Use the Fundamental Theorem to compute

   $$
   \int_0^2(3x^2+2)\,dx.
   $$

3. Explain the difference between signed area and total geometric area.

4. Why does the constant of integration cancel in a definite integral?


In [ ]:
reveal_markdown("Open Section 2 solutions", r"""
1. The width is

   $$
   \Delta x=\frac{1}{n},
   $$

   and the right endpoint is

   $$
   x_i=\frac{i}{n}.
   $$

   Thus

   $$
   S_n=\sum_{i=1}^{n}\frac{i}{n}\frac{1}{n}
   =\frac{1}{n^2}\sum_{i=1}^{n}i.
   $$

   Using

   $$
   \sum_{i=1}^{n}i=\frac{n(n+1)}{2},
   $$

   gives

   $$
   S_n=\frac{n+1}{2n}\longrightarrow \frac{1}{2}.
   $$

2. An antiderivative is

   $$
   F(x)=x^3+2x.
   $$

   Therefore

   $$
   \int_0^2(3x^2+2)\,dx
   =F(2)-F(0)
   =(8+4)-0
   =\boxed{12}.
   $$

3. Signed area subtracts area below the horizontal axis. Total geometric area counts every region positively, which can be computed by integrating the absolute value or splitting the interval.

4. Endpoint evaluation produces

   $$
   (F(b)+C)-(F(a)+C)=F(b)-F(a).
   $$
""")


---

# Section 3: Improper Integrals and Infinite Ranges

## 1. Why an integral can be improper

An integral is **improper** when:

- an interval extends to infinity; or
- the integrand becomes unbounded within the interval.

An improper integral is defined using a limit. For an infinite upper endpoint,

$$
\int_a^\infty f(x)\,dx
=\lim_{b\to\infty}\int_a^b f(x)\,dx.
$$

If this limit exists and is finite, the integral **converges**. Otherwise, it **diverges**.


## 2. Exponential decay over an infinite interval

Compute

$$
\int_0^\infty e^{-x}\,dx.
$$

Replace infinity by a finite endpoint:

$$
\int_0^\infty e^{-x}\,dx
=\lim_{b\to\infty}\int_0^b e^{-x}\,dx.
$$

An antiderivative is

$$
-e^{-x}.
$$

Therefore

$$
\int_0^b e^{-x}\,dx
=\left[-e^{-x}\right]_0^b
=-e^{-b}-(-e^0)
=1-e^{-b}.
$$

Since

$$
\lim_{b\to\infty}e^{-b}=0,
$$

we obtain

$$
\boxed{\int_0^\infty e^{-x}\,dx=1.}
$$

An infinitely long interval can have finite area because the height decays quickly enough.


In [ ]:
bounds_improper = np.linspace(0, 8, 200)
partial_area_improper = 1-np.exp(-bounds_improper)
fig, ax = plt.subplots()
ax.plot(bounds_improper, partial_area_improper, linewidth=2)
ax.axhline(1, color="crimson", linestyle="--", label="limiting area = 1")
ax.set_xlabel("finite upper bound b")
ax.set_ylabel("integral from 0 to b")
ax.set_title("Partial exponential areas converge to one")
ax.legend()
plt.show()


## 3. The p-integral benchmark

Consider

$$
\int_1^\infty \frac{1}{x^p}\,dx.
$$

For $$p\ne 1,$$ an antiderivative is

$$
\frac{x^{1-p}}{1-p}.
$$

If $$p>1,$$ then $$1-p<0$$ and

$$
\lim_{b\to\infty}b^{1-p}=0.
$$

Thus

$$
\int_1^\infty \frac{1}{x^p}\,dx
=\frac{1}{p-1}.
$$

If $$p=1,$$ then

$$
\int_1^b\frac{1}{x}\,dx=\ln b,
$$

which grows without bound. If $$p<1,$$ the power term also grows without bound. Therefore

$$
\boxed{\int_1^\infty x^{-p}\,dx
\text{ converges exactly when }p>1.}
$$


## 4. An unbounded integrand

The integral

$$
\int_0^1\frac{1}{\sqrt{x}}\,dx
$$

is improper because the integrand is unbounded at zero. Define it by

$$
\lim_{a\to 0^+}\int_a^1x^{-1/2}\,dx.
$$

Since

$$
\int x^{-1/2}\,dx=2x^{1/2},
$$

we have

$$
\int_a^1x^{-1/2}\,dx
=2-2\sqrt{a}.
$$

Taking the limit gives

$$
\boxed{\int_0^1\frac{1}{\sqrt{x}}\,dx=2.}
$$

An unbounded height does not automatically cause divergence; the width near the singular point also matters.


> **Common mistakes in Section 3**
>
> - Substituting infinity as though it were an ordinary number.
> - Declaring an integral convergent because the integrand approaches zero.
> - Ignoring an interior or endpoint singularity.
> - Forgetting to split an integral around an interior singularity.
> - Confusing an unbounded interval with an automatically infinite area.

## Section 3 terminology

| Term | Meaning |
|---|---|
| Improper integral | Integral with an infinite interval or unbounded integrand |
| Convergent integral | Improper integral whose defining limit is finite |
| Divergent integral | Improper integral whose defining limit is not finite |
| Tail | Portion of an infinite-range integral beyond a finite point |
| Singularity | Point where an integrand is undefined or unbounded |
| Decay rate | Speed at which a function approaches zero |


## Section 3 guided worksheet

1. Evaluate

   $$
   \int_0^\infty 2e^{-2x}\,dx.
   $$

2. Determine whether

   $$
   \int_1^\infty \frac{1}{x^{3/2}}\,dx
   $$

   converges, and find its value.

3. Determine whether

   $$
   \int_1^\infty \frac{1}{x}\,dx
   $$

   converges.

4. Explain why the condition $$f(x)\to 0$$ is necessary but not sufficient for a positive infinite-range integral to converge.


In [ ]:
reveal_markdown("Open Section 3 solutions", r"""
1. Use a finite upper bound:

   $$
   \int_0^b2e^{-2x}\,dx
   =\left[-e^{-2x}\right]_0^b
   =1-e^{-2b}.
   $$

   Hence

   $$
   \int_0^\infty2e^{-2x}\,dx
   =\lim_{b\to\infty}(1-e^{-2b})
   =\boxed{1}.
   $$

2. Since $$p=3/2>1,$$ the integral converges. Directly,

   $$
   \int_1^b x^{-3/2}\,dx
   =\left[-2x^{-1/2}\right]_1^b
   =2-\frac{2}{\sqrt{b}}.
   $$

   Thus the value is

   $$
   \boxed{2}.
   $$

3. We have

   $$
   \int_1^b\frac{1}{x}\,dx=\ln b,
   $$

   and $$\ln b$$ grows without bound. The integral diverges.

4. The function $$1/x$$ approaches zero but its accumulated tail area is infinite. The speed of decay, not merely the limiting height, determines convergence.
""")


---

# Section 4: Continuous Random Variables, PDFs, and CDFs

## 1. Continuous random variables

A **random variable** assigns a numerical value to a random outcome. A random variable is modeled as **continuous** when it can take values throughout intervals, such as response time, measurement error, or a sensor reading.

A **probability density function**, or **PDF**, is a function $$f_X$$ satisfying:

$$
f_X(x)\geq 0
$$

for every $$x,$$ and

$$
\int_{-\infty}^{\infty}f_X(x)\,dx=1.
$$

Probability over an interval is area under the density:

$$
P(a\leq X\leq b)=\int_a^b f_X(x)\,dx.
$$


## 2. Density is not point probability

For a continuous random variable,

$$
P(X=x)=0
$$

at every individual point, even when $$f_X(x)>1.$$ Density measures probability concentration per unit of the horizontal variable. It can exceed one as long as its total integral is one.

For example, the uniform density on the short interval from zero to one-half is

$$
f_X(x)=2,
\qquad 0\leq x\leq \frac{1}{2}.
$$

Its total probability is

$$
\int_0^{1/2}2\,dx=2\left(\frac{1}{2}\right)=1.
$$

The height is two, but no probability equals two.


## 3. Exponential distribution: normalization and probabilities

For rate $$\lambda>0,$$ the exponential PDF is

$$
f_X(x)=
\begin{cases}
\lambda e^{-\lambda x},&x\geq 0,\\
0,&x<0.
\end{cases}
$$

Verify normalization:

$$
\int_{-\infty}^{\infty}f_X(x)\,dx
=\int_0^\infty \lambda e^{-\lambda x}\,dx.
$$

An antiderivative is

$$
-e^{-\lambda x}.
$$

Therefore

$$
\int_0^\infty \lambda e^{-\lambda x}\,dx
=\lim_{b\to\infty}\left[-e^{-\lambda x}\right]_0^b
=\lim_{b\to\infty}(1-e^{-\lambda b})
=1.
$$

For $$\lambda=2,$$ the probability of a value between zero and one is

$$
P(0\leq X\leq 1)
=\int_0^1 2e^{-2x}\,dx
=1-e^{-2}
\approx 0.864665.
$$


In [ ]:
lambda_exp = 2.0
x_exp = np.linspace(0, 3, 500)
pdf_exp = lambda_exp*np.exp(-lambda_exp*x_exp)
mask_exp = x_exp <= 1

fig, ax = plt.subplots()
ax.plot(x_exp, pdf_exp, linewidth=2, label="exponential density")
ax.fill_between(x_exp[mask_exp], pdf_exp[mask_exp], alpha=0.35,
                label="probability from 0 to 1")
ax.set_xlabel("x")
ax.set_ylabel("density")
ax.set_title("Probability is area under a PDF")
ax.legend()
plt.show()


## 4. Cumulative distribution function

The **cumulative distribution function**, or **CDF**, is

$$
F_X(x)=P(X\leq x).
$$

For a continuous random variable with PDF $$f_X,$$

$$
F_X(x)=\int_{-\infty}^{x}f_X(t)\,dt.
$$

Where differentiable,

$$
F_X'(x)=f_X(x).
$$

For an exponential distribution,

$$
F_X(x)=
\begin{cases}
0,&x<0,\\
1-e^{-\lambda x},&x\geq 0.
\end{cases}
$$

Therefore interval probabilities can also be computed by subtraction:

$$
P(a\leq X\leq b)=F_X(b)-F_X(a).
$$


In [ ]:
cdf_exp = 1-np.exp(-lambda_exp*x_exp)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(x_exp, pdf_exp, linewidth=2)
axes[0].set_xlabel("x")
axes[0].set_ylabel("density")
axes[0].set_title("PDF: local probability concentration")
axes[1].plot(x_exp, cdf_exp, linewidth=2, color="darkgreen")
axes[1].axhline(1, color="gray", linestyle="--")
axes[1].set_xlabel("x")
axes[1].set_ylabel("cumulative probability")
axes[1].set_title("CDF: accumulated probability")
plt.tight_layout()
plt.show()


## 5. Gaussian distribution

The Gaussian, or normal, PDF with mean $$\mu$$ and standard deviation $$\sigma>0$$ is

$$
f_X(x)=\frac{1}{\sigma\sqrt{2\pi}}
\exp\left(-\frac{(x-\mu)^2}{2\sigma^2}\right).
$$

The parameter $$\mu$$ controls the center, and $$\sigma$$ controls the spread. The normalization factor ensures total area one.

Gaussian random values are often used in parameter initialization and noise models. The choice of scale matters because activations can become too small or too large as signals pass through many layers.

Unlike polynomials or exponentials of a linear argument, the Gaussian density has no elementary antiderivative. Its probabilities are evaluated numerically or through standard software functions.


In [ ]:
x_gauss = np.linspace(-5, 5, 600)
fig, ax = plt.subplots()
for sigma_gauss in [0.5, 1.0, 2.0]:
    pdf_gauss = np.exp(-x_gauss**2/(2*sigma_gauss**2))/(sigma_gauss*np.sqrt(2*np.pi))
    ax.plot(x_gauss, pdf_gauss, linewidth=2, label=f"standard deviation = {sigma_gauss}")
ax.set_xlabel("x")
ax.set_ylabel("density")
ax.set_title("Gaussian scale changes height and spread, not total area")
ax.legend()
plt.show()


> **Common mistakes in Section 4**
>
> - Treating a PDF value as a probability.
> - Allowing a PDF to be negative.
> - Forgetting that the total PDF area must equal one.
> - Confusing a PDF with a CDF.
> - Assuming every continuous distribution has an elementary CDF formula.
> - Interpreting the Gaussian standard deviation as its total width.

## Section 4 terminology

| Term | Meaning |
|---|---|
| Random variable | Numerical quantity determined by a random outcome |
| Continuous random variable | Variable modeled over intervals of real values |
| PDF | Nonnegative density whose total integral is one |
| Probability density | Probability concentration per unit of the variable |
| CDF | Probability accumulated up to a specified value |
| Support | Set of values where a distribution can place probability |
| Exponential distribution | Nonnegative distribution with exponential tail decay |
| Gaussian distribution | Bell-shaped distribution specified by a mean and standard deviation |


## Section 4 guided worksheet

Let

$$
f_X(x)=
\begin{cases}
cx,&0\leq x\leq 2,\\
0,&\text{otherwise}.
\end{cases}
$$

1. Find $$c$$ so that $$f_X$$ is a valid PDF.
2. Compute $$P(0.5\leq X\leq 1.5).$$
3. Find the CDF for $$0\leq x\leq 2.$$
4. Explain why $$f_X(1)$$ is not the probability that $$X=1.$$


In [ ]:
reveal_markdown("Open Section 4 solutions", r"""
1. Normalization requires

   $$
   \int_0^2cx\,dx=1.
   $$

   Therefore

   $$
   c\left[\frac{x^2}{2}\right]_0^2
   =c\left(\frac{4}{2}\right)
   =2c=1.
   $$

   Hence

   $$
   \boxed{c=\frac{1}{2}}.
   $$

2. Compute

   $$
   P(0.5\leq X\leq 1.5)
   =\int_{0.5}^{1.5}\frac{x}{2}\,dx.
   $$

   Thus

   $$
   P=\left[\frac{x^2}{4}\right]_{0.5}^{1.5}
   =\frac{2.25-0.25}{4}
   =\boxed{0.5}.
   $$

3. For $$0\leq x\leq 2,$$

   $$
   F_X(x)=\int_0^x\frac{t}{2}\,dt
   =\boxed{\frac{x^2}{4}}.
   $$

4. A point has zero width, so its area under a continuous density is zero. The value $$f_X(1)=1/2$$ is a density height, not point probability.
""")


---

# Section 5: Expected Values and Expected Loss

## 1. Expected value as a probability-weighted average

For a continuous random variable $$X$$ with PDF $$f_X,$$ the expected value of a function $$g(X)$$ is

$$
\boxed{E[g(X)]=\int_{-\infty}^{\infty}g(x)f_X(x)\,dx,}
$$

when the integral exists.

The mean is the special case

$$
E[X]=\int_{-\infty}^{\infty}x f_X(x)\,dx.
$$

This is a probability-weighted average over all possible values, not necessarily a value that is actually observed.


## 2. Detailed uniform example

Let $$X$$ be uniform on the interval from zero to two. Its density is

$$
f_X(x)=\frac{1}{2},
\qquad 0\leq x\leq 2.
$$

The mean is

$$
E[X]=\int_0^2x\left(\frac{1}{2}\right)\,dx.
$$

Pull out the constant:

$$
E[X]=\frac{1}{2}\int_0^2x\,dx.
$$

Evaluate:

$$
E[X]=\frac{1}{2}\left[\frac{x^2}{2}\right]_0^2
=\frac{1}{2}\left(\frac{4}{2}\right)
=\boxed{1}.
$$

The second moment is

$$
E[X^2]=\int_0^2x^2\left(\frac{1}{2}\right)\,dx
=\frac{1}{2}\left[\frac{x^3}{3}\right]_0^2
=\boxed{\frac{4}{3}}.
$$


## 3. Exponential mean by integration by parts

For rate $$\lambda>0,$$

$$
E[X]=\int_0^\infty x\lambda e^{-\lambda x}\,dx.
$$

Use integration by parts:

$$
\int u\,dv=uv-\int v\,du.
$$

Choose

$$
u=x,
\qquad
dv=\lambda e^{-\lambda x}\,dx.
$$

Then

$$
du=dx,
\qquad
v=-e^{-\lambda x}.
$$

For a finite upper bound $$b,$$

$$
\int_0^b x\lambda e^{-\lambda x}\,dx
=\left[-xe^{-\lambda x}\right]_0^b
+\int_0^b e^{-\lambda x}\,dx.
$$

The first term approaches zero because exponential decay dominates linear growth. The remaining integral is

$$
\int_0^\infty e^{-\lambda x}\,dx=\frac{1}{\lambda}.
$$

Therefore

$$
\boxed{E[X]=\frac{1}{\lambda}.}
$$


## 4. Expected loss in machine learning

Let $$Z=(X,Y)$$ represent a random input-target pair drawn from a population distribution. If a model with parameter $$w$$ incurs loss

$$
\ell(w;Z),
$$

then the ideal **population risk** is

$$
R(w)=E[\ell(w;Z)].
$$

For a continuous density $$p(z),$$

$$
R(w)=\int \ell(w;z)p(z)\,dz.
$$

This expression averages loss over all possible data, weighted by how likely those data are.

Because the true population distribution is usually unknown, training uses the **empirical risk** from observed samples:

$$
\widehat{R}_n(w)=\frac{1}{n}\sum_{i=1}^{n}\ell(w;z_i).
$$

The finite average estimates the population expectation.


## 5. Expected squared loss for a constant predictor

Suppose

$$
Y\sim \operatorname{Uniform}(0,1)
$$

and the model always predicts a constant $$a.$$ The squared loss is

$$
\ell(a;Y)=(a-Y)^2.
$$

The expected loss is

$$
R(a)=E[(a-Y)^2]
=\int_0^1(a-y)^2\,dy.
$$

Expand:

$$
(a-y)^2=a^2-2ay+y^2.
$$

Integrate term by term with respect to $$y$$:

$$
R(a)=a^2\int_0^1dy
-2a\int_0^1y\,dy
+\int_0^1y^2\,dy.
$$

Evaluate the three integrals:

$$
\int_0^1dy=1,
$$

$$
\int_0^1y\,dy=\frac{1}{2},
$$

and

$$
\int_0^1y^2\,dy=\frac{1}{3}.
$$

Therefore

$$
R(a)=a^2-a+\frac{1}{3}.
$$

Differentiate with respect to the model parameter:

$$
R'(a)=2a-1.
$$

Set the derivative to zero:

$$
2a-1=0
\quad\Longrightarrow\quad
\boxed{a=\frac{1}{2}.}
$$

Since

$$
R''(a)=2>0,
$$

the expected squared loss is minimized by the population mean.


In [ ]:
a_expected = np.linspace(-0.5, 1.5, 400)
risk_expected = a_expected**2-a_expected+1/3
fig, ax = plt.subplots()
ax.plot(a_expected, risk_expected, linewidth=2)
ax.scatter([0.5], [1/12], color="crimson", s=80, zorder=3,
           label="minimum at population mean")
ax.set_xlabel("constant prediction a")
ax.set_ylabel("expected squared loss")
ax.set_title("Expected loss averages over all possible targets")
ax.legend()
plt.show()


> **Common mistakes in Section 5**
>
> - Omitting the PDF from an expectation integral.
> - Confusing an expected value with the most likely value.
> - Assuming the population distribution is known from a finite sample.
> - Treating empirical risk as exactly equal to population risk.
> - Integrating with respect to the model parameter instead of the random variable.
> - Assuming every expectation is finite.

## Section 5 terminology

| Term | Meaning |
|---|---|
| Expected value | Probability-weighted average over possible values |
| Mean | Expected value of the random variable itself |
| Moment | Expected value of a power of a random variable |
| Loss | Penalty assigned to one prediction-outcome pair |
| Population risk | Expected loss under the data-generating distribution |
| Empirical risk | Average loss over observed samples |
| Generalization | Performance on new data from the population |


## Section 5 guided worksheet

Let $$Y$$ be uniform on the interval from zero to two, and let a model always predict $$a.$$

1. Write the PDF of $$Y.$$
2. Set up the expected absolute loss

   $$
   E[|a-Y|].
   $$

3. For $$a=1,$$ compute the expected absolute loss by splitting the integral at $$y=1.$$
4. Explain the difference between this expectation and the average absolute loss over ten observed targets.


In [ ]:
reveal_markdown("Open Section 5 solutions", r"""
1. The density is

   $$
   f_Y(y)=\frac{1}{2},
   \qquad 0\leq y\leq 2.
   $$

2. The expected absolute loss is

   $$
   E[|a-Y|]=\int_0^2|a-y|\frac{1}{2}\,dy.
   $$

3. At $$a=1,$$ split where the absolute-value expression changes sign:

   $$
   E[|1-Y|]
   =\frac{1}{2}\int_0^1(1-y)\,dy
   +\frac{1}{2}\int_1^2(y-1)\,dy.
   $$

   The first integral is

   $$
   \left[y-\frac{y^2}{2}\right]_0^1
   =1-\frac{1}{2}=\frac{1}{2}.
   $$

   The second is

   $$
   \left[\frac{y^2}{2}-y\right]_1^2
   =0-\left(\frac{1}{2}-1\right)
   =\frac{1}{2}.
   $$

   Multiplying both by one-half gives

   $$
   E[|1-Y|]=\frac{1}{4}+\frac{1}{4}
   =\boxed{\frac{1}{2}}.
   $$

4. The expectation averages over the entire population distribution. The ten-observation average is a sample estimate and changes when a different sample is drawn.
""")


---

# Section 6: Approximating Integrals from Grids and Samples

## 1. Two different approximation problems

Integration in AI appears in at least two computational forms:

1. **Quadrature:** function values are evaluated on a designed grid and combined to approximate an integral.
2. **Monte Carlo estimation:** random samples are drawn and averaged to approximate an expectation.

For

$$
E[g(X)]=\int g(x)f_X(x)\,dx,
$$

the Monte Carlo estimator is

$$
\frac{1}{n}\sum_{i=1}^{n}g(X_i),
$$

where the $$X_i$$ are sampled from the distribution of $$X.$$

Quadrature error depends on grid resolution and smoothness. Monte Carlo error depends on random sample variation and sample size.


## 2. Numerical demonstration

For

$$
X\sim \operatorname{Uniform}(0,1),
$$

the exact expectation is

$$
E[X^2]=\int_0^1x^2\,dx=\frac{1}{3}.
$$

We compare a uniform-grid trapezoidal approximation with random sample averages.


In [ ]:
rng_integral = np.random.default_rng(2026)
exact_second_moment = 1/3

grid_sizes_mc = [5, 10, 50, 100, 1000]
quadrature_estimates = []
monte_carlo_estimates = []
for size in grid_sizes_mc:
    grid = np.linspace(0, 1, size)
    quadrature_estimates.append(trapezoid_integral(grid, grid**2))
    sample = rng_integral.uniform(0, 1, size)
    monte_carlo_estimates.append(np.mean(sample**2))

print("Exact value:", exact_second_moment)
for size, quad, mc in zip(grid_sizes_mc, quadrature_estimates, monte_carlo_estimates):
    print(f"size = {size:4d}, quadrature = {quad:.6f}, Monte Carlo = {mc:.6f}")


## 3. Batch loss as a Monte Carlo estimate

When training examples are modeled as random draws from a population, a minibatch loss

$$
\widehat{R}_B(w)=\frac{1}{B}\sum_{i=1}^{B}\ell(w;Z_i)
$$

is a Monte Carlo estimate of

$$
R(w)=E[\ell(w;Z)].
$$

Its derivative, when differentiation and expectation can be interchanged, estimates the population-risk derivative:

$$
\frac{d}{dw}R(w)
=E\left[\frac{\partial}{\partial w}\ell(w;Z)\right].
$$

This provides a mathematical foundation for stochastic gradient methods. A small batch is computationally cheap but produces a noisier estimate; a large batch is more stable but more expensive.


> **Common mistakes in Section 6**
>
> - Treating grid resolution and random sample size as identical ideas.
> - Assuming one Monte Carlo run must approach the exact value monotonically.
> - Forgetting to sample from the distribution appearing in the expectation.
> - Reporting a sample estimate without acknowledging sampling variation.
> - Assuming a larger batch automatically improves generalization.

## Section 6 terminology

| Term | Meaning |
|---|---|
| Quadrature | Deterministic numerical approximation of an integral |
| Monte Carlo estimate | Random-sample average approximating an expectation |
| Sampling variation | Change in an estimate across random samples |
| Minibatch loss | Average loss on a subset used for one update |
| Stochastic gradient | Random-sample estimate of a population gradient |
| Grid resolution | Fineness of deterministic integration points |


## Section 6 guided worksheet

1. Write a trapezoidal approximation for

   $$
   \int_0^1e^{-x^2}\,dx
   $$

   using grid points $$0,$$ $$0.5,$$ and $$1.$$

2. If samples $$0.2,$$ $$0.6,$$ and $$0.9$$ are drawn from a uniform distribution on zero to one, estimate $$E[X^2]$$ by Monte Carlo averaging.

3. State one source of error for each method.
4. Explain why two minibatches can produce different gradient estimates at the same parameter value.


In [ ]:
reveal_markdown("Open Section 6 solutions", r"""
1. Each width is $$0.5.$$ The trapezoidal approximation is

   $$
   \frac{0.5}{2}\left(e^0+e^{-0.25}\right)
   +\frac{0.5}{2}\left(e^{-0.25}+e^{-1}\right).
   $$

   Equivalently,

   $$
   \boxed{0.25\left(1+2e^{-0.25}+e^{-1}\right)}.
   $$

2. The estimate is

   $$
   \frac{0.2^2+0.6^2+0.9^2}{3}
   =\frac{0.04+0.36+0.81}{3}
   =\frac{1.21}{3}
   \approx \boxed{0.403333}.
   $$

3. Trapezoidal error comes from replacing curved segments by straight segments on a finite grid. Monte Carlo error comes from random sampling variation in a finite sample.

4. Different minibatches contain different examples, so their average losses and derivatives are different random estimates of the same population quantities.
""")


---

# Integrated Instructor Lab: Optimal Prediction Under Expected Squared Loss

Suppose a target has density

$$
f_Y(y)=2y,
\qquad 0\leq y\leq 1.
$$

First verify normalization:

$$
\int_0^1 2y\,dy
=\left[y^2\right]_0^1=1.
$$

A model predicts a constant $$a,$$ and its squared loss is

$$
\ell(a;y)=(a-y)^2.
$$

The population risk is

$$
R(a)=\int_0^1(a-y)^2(2y)\,dy.
$$

Expand the integrand:

$$
(a-y)^2(2y)
=(a^2-2ay+y^2)(2y)
=2a^2y-4ay^2+2y^3.
$$

Integrate:

$$
R(a)
=2a^2\left[\frac{y^2}{2}\right]_0^1
-4a\left[\frac{y^3}{3}\right]_0^1
+2\left[\frac{y^4}{4}\right]_0^1.
$$

Therefore

$$
R(a)=a^2-\frac{4}{3}a+\frac{1}{2}.
$$

Differentiate:

$$
R'(a)=2a-\frac{4}{3}.
$$

The critical point is

$$
2a-\frac{4}{3}=0
\quad\Longrightarrow\quad
\boxed{a=\frac{2}{3}}.
$$

Because $$R''(a)=2>0,$$ this is the global minimum. It also equals the population mean:

$$
E[Y]=\int_0^1y(2y)\,dy
=2\left[\frac{y^3}{3}\right]_0^1
=\frac{2}{3}.
$$


In [ ]:
a_grid_lab6 = np.linspace(-0.2, 1.2, 400)
risk_grid_lab6 = a_grid_lab6**2-(4/3)*a_grid_lab6+0.5

rng_lab6 = np.random.default_rng(7)
sample_y_lab6 = np.sqrt(rng_lab6.uniform(0, 1, 40))
empirical_risk_lab6 = np.array([
    np.mean((a-sample_y_lab6)**2) for a in a_grid_lab6
])
empirical_optimum_lab6 = np.mean(sample_y_lab6)

fig, ax = plt.subplots()
ax.plot(a_grid_lab6, risk_grid_lab6, linewidth=2, label="population expected loss")
ax.plot(a_grid_lab6, empirical_risk_lab6, linestyle="--", linewidth=2,
        label="empirical loss from 40 samples")
ax.axvline(2/3, color="black", linestyle=":", label="population optimum")
ax.axvline(empirical_optimum_lab6, color="crimson", linestyle=":",
           label="sample optimum")
ax.set_xlabel("constant prediction a")
ax.set_ylabel("squared loss")
ax.set_title("Empirical risk approximates population risk")
ax.legend()
plt.show()


## Integrated lab tasks

1. Verify the PDF normalization and expected value independently.
2. Differentiate the expected loss before evaluating the integrals and show that the same optimum results.
3. Explain why the sample optimum in the graph need not equal exactly two-thirds.
4. Predict how the empirical-loss curve should change as the sample size becomes very large.
5. Explain why the best constant predictor under squared loss is the mean.


In [ ]:
reveal_markdown("Open integrated-lab solution", r"""
Differentiate under the integral sign:

$$
R'(a)=\int_0^1\frac{\partial}{\partial a}(a-y)^2(2y)\,dy.
$$

Since

$$
\frac{\partial}{\partial a}(a-y)^2=2(a-y),
$$

we get

$$
R'(a)=\int_0^1 2(a-y)(2y)\,dy.
$$

Separate the terms:

$$
R'(a)=2a\int_0^1 2y\,dy
-2\int_0^1y(2y)\,dy.
$$

The first integral equals one and the second is $$E[Y]=2/3.$$ Hence

$$
R'(a)=2a-2\left(\frac{2}{3}\right)
=2a-\frac{4}{3}.
$$

Setting this equal to zero gives $$a=2/3.$$

The sample optimum is the sample mean, which varies from one random sample to another. As sample size grows, the empirical curve and optimum generally become more stable and approach their population counterparts. In general,

$$
\frac{d}{da}E[(a-Y)^2]=2(a-E[Y]),
$$

so the unique stationary point is $$a=E[Y],$$ with positive second derivative.
""")


---

# Module 6 Summary

1. Indefinite integrals describe families of antiderivatives.
2. Definite integrals arise as limits of sums and measure signed accumulation.
3. The Fundamental Theorem connects derivatives and integrals.
4. Improper integrals use limits to handle infinite ranges and singularities.
5. A continuous PDF is nonnegative and integrates to one.
6. Probability is PDF area, while a CDF is accumulated probability.
7. Expected value is a density-weighted integral.
8. Population risk is expected loss; empirical risk is its finite-sample estimate.

## Cumulative terminology

| AI or probability language | Integration interpretation |
|---|---|
| PDF normalization | Total density integral equals one |
| Interval probability | Area under the PDF over that interval |
| CDF | Integral of the PDF up to a threshold |
| Expected loss | Loss integrated against the data distribution |
| Empirical loss | Sample average approximating expected loss |
| Minibatch estimate | Monte Carlo approximation from a subset |
| Continuous support | Range requiring integration rather than point summation |

## Exit ticket

1. Why can a PDF be greater than one while a probability cannot?
2. Why is infinity replaced by a limit in an improper integral?
3. What is the difference between a PDF and a CDF?
4. What does a population risk integral average?
5. Why will empirical risk generally differ from population risk?
